In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt


In [ ]:

DATA_FILE = '../data/scenario_6_marina_features_50ms_final.csv'
df = pd.read_csv(DATA_FILE)

STEADY_THRESHOLD = 10000  # 10 seconds of buffer is healthy
DEPLETING_THRESHOLD = 1000 # Below 1 seconds is a high risk of stalling

print("Dataset loaded successfully.")
df.info()

In [ ]:

def assign_buffer_state(bh_ms):
    if bh_ms < DEPLETING_THRESHOLD:
        return 'Depleting'
    elif bh_ms > STEADY_THRESHOLD:
        return 'Steady'
    else:
        return 'Filling'

df['buffer_state'] = df['buffer_level_ms'].apply(assign_buffer_state)

print("Class Distribution in the Dataset:")
print(df['buffer_state'].value_counts())

In [ ]:
all_features = [
    'packet_count', 
    'ps_sum', 'ps2_sum', 'ps3_sum', 
    'iat_sum', 'iat2_sum', 'iat3_sum'
]
X = df[all_features]
y = df['buffer_state']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

dt_baseline = DecisionTreeClassifier(
    max_depth=5, 
    random_state=42, 
    class_weight='balanced' 
)
dt_baseline.fit(X_train, y_train)

print("--- Baseline Model (All Marina Features) Evaluation ---")
y_pred_baseline = dt_baseline.predict(X_test)
print(classification_report(y_test, y_pred_baseline))

In [ ]:
importances = pd.Series(dt_baseline.feature_importances_, index=all_features)
importances.sort_values(ascending=False, inplace=True)

print("\n--- Feature Importances from Baseline Model ---")
print(importances)

plt.figure(figsize=(10, 6))
importances.sort_values().plot(kind='barh', title='Feature Importances for Marina Features')
plt.show()

In [ ]:
plt.figure(figsize=(30, 18))
plot_tree(dt_baseline, 
          feature_names=all_features, 
          class_names=sorted(y.unique()),
          filled=True, 
          rounded=True, 
          fontsize=10)
plt.title("Decision Tree Visualization (Balanced Model with Marina Features)")
plt.show()

In [ ]:
feature_experiments = {
    "First_Moment_Only": ['packet_count', 'ps_sum', 'iat_sum'],
    "Packet_Size_Features_Only": ['packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum'],
    "IAT_Features_Only": ['packet_count', 'iat_sum', 'iat2_sum', 'iat3_sum'],
    "All_Marina_Features": all_features
}

results = {}

print("\n\n--- Running Feature Set Experiments ---")

for name, features_to_test in feature_experiments.items():
    print(f"\n--- Testing: {name} ---")
    
    X_exp = df[features_to_test]
    y_exp = df['buffer_state']
    X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
        X_exp, y_exp, test_size=0.25, random_state=42, stratify=y_exp
    )
    
    model = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
    model.fit(X_train_exp, y_train_exp)
    
    y_pred_exp = model.predict(X_test_exp)
    report = classification_report(y_test_exp, y_pred_exp, output_dict=True, zero_division=0)
    
    print(classification_report(y_test_exp, y_pred_exp, zero_division=0))
    
    if 'Depleting' in report:
        results[name] = report['Depleting']['recall']

print("\n\n--- Final Results: Recall for 'Depleting' State Across Experiments ---")
results_series = pd.Series(results)
results_series.sort_values(ascending=False, inplace=True)
print(results_series)